# Lab 6 — Tools & Tool Calling with LLMs

This notebook covers all 10 exercises of Lab 6 with **LangChain**. Every exercise follows the same tool-calling flow:

1. **Define** the tool(s): `@tool`, `StructuredTool` + Pydantic `args_schema`, or a `BaseTool` subclass.
2. **Bind** them to the chat model with `llm.bind_tools([...])`.
3. **Ask** a natural-language question and print the model's suggested `tool_calls`.
4. **Execute** each suggested call manually and wrap the result in a `ToolMessage`.
5. **Feed** the results back and print the model's final natural-language answer.

### How to run (Windows + VS Code)
1. Install the VS Code **Python** and **Jupyter** extensions and use Python 3.10 or newer.
2. Open this file and choose your Python interpreter as the kernel (top-right *Select Kernel*).
3. Get a free Groq API key from <https://console.groq.com/keys>. Either create a file named `.env` next to this notebook containing `GROQ_API_KEY=gsk_...`, or just run the setup cell and paste the key when prompted.
4. Run the cells **from top to bottom** (Shift+Enter). The setup section must run before any exercise.

> To use Gemini, OpenAI or a local Ollama model instead, change `PROVIDER` in the *Choose the LLM* cell.

## 0. Setup

In [1]:
# Install the required packages (only needed the first time)
%pip install -qU langchain-core langchain-groq pandas python-dotenv

# Optional: install only if you switch PROVIDER below
# %pip install -qU langchain-google-genai    # PROVIDER = "gemini"
# %pip install -qU langchain-openai          # PROVIDER = "openai"
# %pip install -qU langchain-ollama          # PROVIDER = "ollama"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, time, asyncio, getpass
from datetime import datetime
from typing import Literal, Optional, Type

import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from dotenv import load_dotenv

from langchain_core.tools import tool, StructuredTool, BaseTool, BaseToolkit
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
from langchain_core.callbacks import CallbackManagerForToolRun, AsyncCallbackManagerForToolRun

load_dotenv()          # reads GROQ_API_KEY / GOOGLE_API_KEY / OPENAI_API_KEY from a .env file, if present
pd.set_option("display.max_colwidth", None)
print("Imports OK")

Imports OK


### Choose the LLM
| `PROVIDER` | Default model | API key variable |
|---|---|---|
| `"groq"` (default, free) | `openai/gpt-oss-120b` | `GROQ_API_KEY` |
| `"gemini"` | `gemini-2.5-flash` | `GOOGLE_API_KEY` |
| `"openai"` | `gpt-4o-mini` | `OPENAI_API_KEY` |
| `"ollama"` (local, no key) | `qwen2.5:7b` (run `ollama pull qwen2.5:7b` first) | — |

The model must support tool calling. If your provider has retired a model, set `MODEL` to a current tool-calling model from its model list.

In [3]:
PROVIDER = "groq"      # "groq" | "gemini" | "openai" | "ollama"
MODEL = None           # None = use the default model for the provider

DEFAULT_MODELS = {"groq": "openai/gpt-oss-120b", "gemini": "gemini-2.5-flash",
                  "openai": "gpt-4o-mini", "ollama": "qwen2.5:7b"}
KEY_VARS = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY"}

def get_llm(provider=PROVIDER, model=MODEL):
    model = model or DEFAULT_MODELS[provider]
    if provider in KEY_VARS and not os.environ.get(KEY_VARS[provider]):
        os.environ[KEY_VARS[provider]] = getpass.getpass(f"Enter your {KEY_VARS[provider]}: ")
    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=model, temperature=0)
    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model, temperature=0)
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model, temperature=0)
    if provider == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(model=model, temperature=0)
    raise ValueError(f"Unknown provider: {provider}")

llm = get_llm()
print("Using:", PROVIDER, "->", MODEL or DEFAULT_MODELS[PROVIDER])

Using: groq -> openai/gpt-oss-120b


### Helper functions
These helpers keep every exercise readable:
* `show_tool_calls(ai_msg)` prints the tool calls the model **suggested** (name + arguments).
* `execute_tool_calls(ai_msg, tools, messages)` **manually** runs each suggested call, prints the result and appends a `ToolMessage` to the conversation.
* `get_final_answer(llm_with_tools, messages, tools)` sends the tool results back. If the model asks for more tools, it runs them too, until the model replies in plain text.

In [4]:
def text_of(msg) -> str:
    # Plain text of an AI message (Gemini can return a list of content parts)
    c = msg.content
    if isinstance(c, list):
        return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in c)
    return c

def show_tool_calls(ai_msg, title="Model's suggested tool_calls"):
    print(f"--- {title} ({len(ai_msg.tool_calls)}) ---")
    if not ai_msg.tool_calls:
        print("  (no tool calls)  text reply:", text_of(ai_msg))
    for i, tc in enumerate(ai_msg.tool_calls, 1):
        print(f"  {i}. {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})   id={tc['id']}")

def execute_tool_calls(ai_msg, tools, messages):
    # Manually execute every suggested tool call; append the results as ToolMessages
    tools_by_name = {t.name: t for t in tools}
    results = []
    print("--- Manually executed tool results ---")
    for tc in ai_msg.tool_calls:
        try:
            result = tools_by_name[tc["name"]].invoke(tc["args"])
        except Exception as e:                       # e.g. Pydantic validation error -> tell the model
            result = {"error": f"{type(e).__name__}: {e}"}
        print(f"  {tc['name']} -> {json.dumps(result, ensure_ascii=False, default=str)}")
        messages.append(ToolMessage(content=json.dumps(result, ensure_ascii=False, default=str),
                                    tool_call_id=tc["id"], name=tc["name"]))
        results.append({"tool": tc["name"], "args": tc["args"], "result": result})
    return results

def get_final_answer(llm_with_tools, messages, tools, max_rounds=5):
    # Feed tool results back; keep executing tools while the model still asks for them
    extra_calls = []
    for r in range(max_rounds):
        ai_msg = llm_with_tools.invoke(messages)
        messages.append(ai_msg)
        if not ai_msg.tool_calls:
            print("=== Final answer ===")
            print(text_of(ai_msg))
            return ai_msg, extra_calls
        show_tool_calls(ai_msg, f"Model asked for more tools (follow-up round {r + 1})")
        extra_calls += execute_tool_calls(ai_msg, tools, messages)
    raise RuntimeError("Model did not produce a final answer within max_rounds")

In [5]:
# Quick connectivity test: the model should answer in one short line
print(text_of(llm.invoke("Reply with exactly: LLM connected")))
# (uses text_of so it also works with Gemini)

LLM connected


---
## Exercise 1 — Order Status Lookup Tool
**Tool method:** `@tool`. This is a simple lookup in an in-memory "orders database", so no strict validation is needed.

In [6]:
ORDERS_DB = {
    "ORD-4521": {"status": "Shipped",          "carrier": "BlueDart",  "current_location": "Pune sorting hub",
                 "expected_delivery": "2026-10-24"},
    "ORD-4522": {"status": "Out for delivery", "carrier": "Delhivery", "current_location": "Mumbai - Andheri",
                 "expected_delivery": "2026-10-21"},
    "ORD-4523": {"status": "Processing",       "carrier": None,        "current_location": "Warehouse",
                 "expected_delivery": "2026-10-27"},
}

@tool
def get_order_status(order_id: str) -> dict:
    # (the docstring below is what the LLM reads)
    '''Look up the current shipping status of a customer order.

    Args:
        order_id: The order ID, e.g. "ORD-4521".
    '''
    order_id = order_id.strip().upper()
    if order_id not in ORDERS_DB:
        return {"order_id": order_id, "found": False, "message": "No order with this ID"}
    return {"order_id": order_id, "found": True, **ORDERS_DB[order_id]}

print("Tool name  :", get_order_status.name)
print("Description:", get_order_status.description)
print("Args schema:", get_order_status.args)
print("Direct call:", get_order_status.invoke({"order_id": "ORD-4521"}))

Tool name  : get_order_status
Description: Look up the current shipping status of a customer order.

    Args:
        order_id: The order ID, e.g. "ORD-4521".
Args schema: {'order_id': {'title': 'Order Id', 'type': 'string'}}
Direct call: {'order_id': 'ORD-4521', 'found': True, 'status': 'Shipped', 'carrier': 'BlueDart', 'current_location': 'Pune sorting hub', 'expected_delivery': '2026-10-24'}


In [7]:
tools_1 = [get_order_status]
llm_with_tools_1 = llm.bind_tools(tools_1)

user_query = "Where is my order ORD-4521?"
messages_1 = [HumanMessage(user_query)]

ai_msg_1 = llm_with_tools_1.invoke(messages_1)
messages_1.append(ai_msg_1)
show_tool_calls(ai_msg_1)

--- Model's suggested tool_calls (1) ---
  1. get_order_status({"order_id": "ORD-4521"})   id=fc_9a2bb71b-f556-47a0-bc12-2e02c2b57ae3


In [8]:
results_1 = execute_tool_calls(ai_msg_1, tools_1, messages_1)

--- Manually executed tool results ---
  get_order_status -> {"order_id": "ORD-4521", "found": true, "status": "Shipped", "carrier": "BlueDart", "current_location": "Pune sorting hub", "expected_delivery": "2026-10-24"}


In [9]:
final_1, _ = get_final_answer(llm_with_tools_1, messages_1, tools_1)

=== Final answer ===
Your order **ORD‑4521** has already been shipped.

- **Carrier:** BlueDart  
- **Current location:** Pune sorting hub  
- **Expected delivery:** **24 Oct 2026**

If you need any additional details (tracking link, delivery instructions, etc.), just let me know!


---
## Exercise 2 — Loan EMI Calculator Tool
**Tool method:** `StructuredTool` + Pydantic `args_schema` with strict validation:
`principal > 0`, `0 ≤ annual_rate ≤ 30`, `tenure_months > 0`.

EMI formula: $EMI = \dfrac{P \cdot r \cdot (1+r)^n}{(1+r)^n - 1}$, where $r$ = monthly rate = annual_rate / 12 / 100 and $n$ = tenure in months.

In [10]:
class EMIInput(BaseModel):
    principal: float = Field(gt=0, description="Loan amount in rupees, e.g. 500000 for 5 lakh")
    annual_rate: float = Field(ge=0, le=30, description="Annual interest rate in percent (0-30), e.g. 9 for 9%")
    tenure_months: int = Field(gt=0, description="Loan tenure in months, e.g. 24")

def calculate_emi(principal: float, annual_rate: float, tenure_months: int) -> dict:
    r = annual_rate / 12 / 100
    n = tenure_months
    emi = principal / n if r == 0 else principal * r * (1 + r) ** n / ((1 + r) ** n - 1)
    total = emi * n
    return {"monthly_emi": round(emi, 2), "total_payment": round(total, 2),
            "total_interest": round(total - principal, 2), "principal": principal,
            "annual_rate_percent": annual_rate, "tenure_months": n}

emi_tool = StructuredTool.from_function(
    func=calculate_emi,
    name="calculate_emi",
    description="Calculate the monthly EMI, total payment and total interest of a loan.",
    args_schema=EMIInput,
)

print("Args schema:", emi_tool.args)
print("Valid call  :", emi_tool.invoke({"principal": 500000, "annual_rate": 9, "tenure_months": 24}))

Args schema: {'principal': {'description': 'Loan amount in rupees, e.g. 500000 for 5 lakh', 'exclusiveMinimum': 0, 'title': 'Principal', 'type': 'number'}, 'annual_rate': {'description': 'Annual interest rate in percent (0-30), e.g. 9 for 9%', 'maximum': 30, 'minimum': 0, 'title': 'Annual Rate', 'type': 'number'}, 'tenure_months': {'description': 'Loan tenure in months, e.g. 24', 'exclusiveMinimum': 0, 'title': 'Tenure Months', 'type': 'integer'}}
Valid call  : {'monthly_emi': 22842.37, 'total_payment': 548216.91, 'total_interest': 48216.91, 'principal': 500000.0, 'annual_rate_percent': 9.0, 'tenure_months': 24}


In [11]:
# Strict validation: each of these inputs must be REJECTED by the Pydantic schema
bad_inputs = [
    {"principal": -500000, "annual_rate": 9,  "tenure_months": 24},   # principal must be > 0
    {"principal": 500000,  "annual_rate": 45, "tenure_months": 24},   # rate must be between 0 and 30
    {"principal": 500000,  "annual_rate": 9,  "tenure_months": 0},    # tenure must be > 0
]
for bad in bad_inputs:
    try:
        emi_tool.invoke(bad)
        print("NOT rejected:", bad)
    except ValidationError as e:
        err = e.errors()[0]
        print(f"Rejected {bad}\n   -> field '{err['loc'][0]}': {err['msg']}")

Rejected {'principal': -500000, 'annual_rate': 9, 'tenure_months': 24}
   -> field 'principal': Input should be greater than 0
Rejected {'principal': 500000, 'annual_rate': 45, 'tenure_months': 24}
   -> field 'annual_rate': Input should be less than or equal to 30
Rejected {'principal': 500000, 'annual_rate': 9, 'tenure_months': 0}
   -> field 'tenure_months': Input should be greater than 0


In [12]:
tools_2 = [emi_tool]
llm_with_tools_2 = llm.bind_tools(tools_2)

user_query = "What's my EMI for a 5 lakh loan at 9% for 24 months?"
messages_2 = [HumanMessage(user_query)]

ai_msg_2 = llm_with_tools_2.invoke(messages_2)
messages_2.append(ai_msg_2)
show_tool_calls(ai_msg_2)

--- Model's suggested tool_calls (1) ---
  1. calculate_emi({"annual_rate": 9, "principal": 500000, "tenure_months": 24})   id=fc_72e208a4-e728-4c76-8cd1-dbb94580defa


In [13]:
results_2 = execute_tool_calls(ai_msg_2, tools_2, messages_2)

--- Manually executed tool results ---
  calculate_emi -> {"monthly_emi": 22842.37, "total_payment": 548216.91, "total_interest": 48216.91, "principal": 500000.0, "annual_rate_percent": 9.0, "tenure_months": 24}


In [14]:
final_2, _ = get_final_answer(llm_with_tools_2, messages_2, tools_2)

=== Final answer ===
Here’s the breakdown for a **₹5 00,000** loan at **9 % p.a.** (reducing balance) over **24 months**:

| Item | Amount (₹) |
|------|------------|
| **Monthly EMI** | **22,842.37** |
| **Total payment over 24 months** | **5,48,216.91** |
| **Total interest paid** | **48,216.91** |

**How it’s calculated**

The monthly interest rate is \( \frac{9\%}{12} = 0.75\% = 0.0075 \).  
Using the standard EMI formula  

\[
EMI = P \times \frac{r(1+r)^n}{(1+r)^n-1}
\]

where  
- \(P = 5,00,000\) (principal)  
- \(r = 0.0075\) (monthly rate)  
- \(n = 24\) (months),

the result is **₹22,842.37** per month. Multiplying by 24 gives the total payment, and subtracting the principal yields the total interest.

Let me know if you’d like to see how the EMI changes with a different tenure or interest rate!


---
## Exercise 3 — Unit Converter Toolkit (Multiple Tools)
**Tool method:** `@tool` for both tools. We bind both to the model and check that it picks **only** the distance tool for a distance question.

In [15]:
@tool
def convert_temperature(value: float, from_unit: Literal["celsius", "fahrenheit"]) -> dict:
    '''Convert a temperature between Celsius and Fahrenheit.

    Args:
        value: The temperature value to convert.
        from_unit: The unit of the given value: "celsius" or "fahrenheit".
    '''
    if from_unit == "celsius":
        return {"input": f"{value} °C", "result": round(value * 9 / 5 + 32, 2), "result_unit": "°F"}
    return {"input": f"{value} °F", "result": round((value - 32) * 5 / 9, 2), "result_unit": "°C"}

@tool
def convert_distance(value: float, from_unit: Literal["km", "miles"]) -> dict:
    '''Convert a distance between kilometres and miles.

    Args:
        value: The distance value to convert.
        from_unit: The unit of the given value: "km" or "miles".
    '''
    KM_PER_MILE = 1.609344
    if from_unit == "km":
        return {"input": f"{value} km", "result": round(value / KM_PER_MILE, 3), "result_unit": "miles"}
    return {"input": f"{value} miles", "result": round(value * KM_PER_MILE, 3), "result_unit": "km"}

print(convert_temperature.invoke({"value": 37, "from_unit": "celsius"}))
print(convert_distance.invoke({"value": 100, "from_unit": "km"}))

{'input': '37.0 °C', 'result': 98.6, 'result_unit': '°F'}
{'input': '100.0 km', 'result': 62.137, 'result_unit': 'miles'}


In [16]:
tools_3 = [convert_temperature, convert_distance]
llm_with_tools_3 = llm.bind_tools(tools_3)          # both tools bound in a single call

user_query = "Convert 100 km to miles"
messages_3 = [HumanMessage(user_query)]

ai_msg_3 = llm_with_tools_3.invoke(messages_3)
messages_3.append(ai_msg_3)
show_tool_calls(ai_msg_3)

picked = [tc["name"] for tc in ai_msg_3.tool_calls]
print("\nTools bound :", [t.name for t in tools_3])
print("Tool picked :", picked)
print("Correct tool chosen?", "YES" if picked == ["convert_distance"] else "NO")

--- Model's suggested tool_calls (1) ---
  1. convert_distance({"from_unit": "km", "value": 100})   id=fc_91a1ce94-b1fc-487f-8e6b-0fc68f975e3e

Tools bound : ['convert_temperature', 'convert_distance']
Tool picked : ['convert_distance']
Correct tool chosen? YES


In [17]:
results_3 = execute_tool_calls(ai_msg_3, tools_3, messages_3)

--- Manually executed tool results ---
  convert_distance -> {"input": "100.0 km", "result": 62.137, "result_unit": "miles"}


In [18]:
final_3, _ = get_final_answer(llm_with_tools_3, messages_3, tools_3)

=== Final answer ===
100 km is approximately **62.14 miles**.


---
## Exercise 4 — Weather-Based Outfit Recommendation (Chained Tools)
**Tool method:** `@tool` for both. `recommend_outfit` needs the output of `get_weather`, so we run the flow in rounds:

1. **Round 1:** the model should call `get_weather`. We execute it.
2. **Round 2:** we feed the weather back. The model should now call `recommend_outfit` and **reuse** the fetched temperature and conditions.
3. We check whether it reused the real values or invented its own, execute the call, and get the final recommendation.

In [19]:
WEATHER_DB = {
    "mumbai":    {"temperature_c": 31, "conditions": "humid and partly cloudy"},
    "pune":      {"temperature_c": 27, "conditions": "light rain"},
    "delhi":     {"temperature_c": 22, "conditions": "hazy"},
    "bangalore": {"temperature_c": 24, "conditions": "cloudy with light breeze"},
    "shimla":    {"temperature_c": 9,  "conditions": "cold and clear"},
}

@tool
def get_weather(city: str) -> dict:
    '''Get the current temperature (in °C) and weather conditions for a city.

    Args:
        city: Name of the city, e.g. "Mumbai".
    '''
    data = WEATHER_DB.get(city.strip().lower())
    if data is None:
        return {"city": city, "error": "Weather data not available for this city"}
    return {"city": city.title(), **data}

@tool
def recommend_outfit(temperature: float, conditions: str) -> dict:
    '''Recommend an outfit for the given weather. Call get_weather first to obtain real values.

    Args:
        temperature: Current temperature in °C.
        conditions: Current weather conditions, e.g. "light rain".
    '''
    c = conditions.lower()
    if temperature >= 30:
        outfit = "Light cotton or linen t-shirt, breathable shorts or chinos, sandals"
    elif temperature >= 22:
        outfit = "Half-sleeve shirt or kurta, cotton trousers, comfortable sneakers"
    elif temperature >= 15:
        outfit = "Full-sleeve shirt with a light jacket or sweatshirt, jeans, closed shoes"
    else:
        outfit = "Thermal inner, sweater, heavy jacket, warm trousers, boots"
    extras = []
    if "rain" in c:                      extras.append("umbrella or raincoat and waterproof footwear")
    if "humid" in c:                     extras.append("sweat-wicking fabric and a water bottle")
    if "sun" in c or "clear" in c:       extras.append("sunglasses and sunscreen")
    if "haz" in c:                       extras.append("an N95 mask")
    return {"temperature": temperature, "conditions": conditions, "outfit": outfit, "extras": extras}

print(get_weather.invoke({"city": "Mumbai"}))
print(recommend_outfit.invoke({"temperature": 31, "conditions": "humid and partly cloudy"}))

{'city': 'Mumbai', 'temperature_c': 31, 'conditions': 'humid and partly cloudy'}
{'temperature': 31.0, 'conditions': 'humid and partly cloudy', 'outfit': 'Light cotton or linen t-shirt, breathable shorts or chinos, sandals', 'extras': ['sweat-wicking fabric and a water bottle']}


In [20]:
tools_4 = [get_weather, recommend_outfit]
llm_with_tools_4 = llm.bind_tools(tools_4)

user_query = "What should I wear today in Mumbai?"
messages_4 = [HumanMessage(user_query)]

# ---------------- Round 1: the model should ask for get_weather ----------------
ai_round1 = llm_with_tools_4.invoke(messages_4)
messages_4.append(ai_round1)
show_tool_calls(ai_round1, "Round 1 - suggested tool_calls")

--- Round 1 - suggested tool_calls (1) ---
  1. get_weather({"city": "Mumbai"})   id=fc_9da03d3b-427d-4a0a-ad2a-2f4354786616


In [21]:
round1_results = execute_tool_calls(ai_round1, tools_4, messages_4)
weather_data = next((r["result"] for r in round1_results if r["tool"] == "get_weather"), None)
print("\nFetched weather data:", weather_data)

--- Manually executed tool results ---
  get_weather -> {"city": "Mumbai", "temperature_c": 31, "conditions": "humid and partly cloudy"}

Fetched weather data: {'city': 'Mumbai', 'temperature_c': 31, 'conditions': 'humid and partly cloudy'}


In [22]:
# ---------------- Round 2: feed the weather back; the model should call recommend_outfit ----------------
outfit_calls = [tc for tc in ai_round1.tool_calls if tc["name"] == "recommend_outfit"]
if outfit_calls:
    print("Note: the model already called recommend_outfit in round 1 (before seeing the weather).")
    ai_round2 = None
else:
    ai_round2 = llm_with_tools_4.invoke(messages_4)
    messages_4.append(ai_round2)
    show_tool_calls(ai_round2, "Round 2 - suggested tool_calls")
    outfit_calls = [tc for tc in ai_round2.tool_calls if tc["name"] == "recommend_outfit"]

# Did the model REUSE the fetched values or INVENT its own?
def _to_float(x):
    try:
        return float(str(x).replace("°C", "").replace("C", "").strip())
    except ValueError:
        return None

for tc in outfit_calls:
    args = tc["args"]
    model_cond, real_cond = str(args.get("conditions", "")).strip().lower(), weather_data["conditions"]
    temp_ok = _to_float(args.get("temperature")) == float(weather_data["temperature_c"])
    cond_ok = model_cond == real_cond or real_cond in model_cond or (model_cond != "" and model_cond in real_cond)
    print("\nCheck recommend_outfit arguments against the fetched weather:")
    print(f"  temperature: model={args.get('temperature')!r:<8} fetched={weather_data['temperature_c']!r:<6} -> {'REUSED' if temp_ok else 'DIFFERENT / INVENTED'}")
    print(f"  conditions : model={args.get('conditions')!r} fetched={weather_data['conditions']!r} -> {'REUSED' if cond_ok else 'DIFFERENT / INVENTED'}")
if not outfit_calls:
    print("The model did not call recommend_outfit.")

--- Round 2 - suggested tool_calls (1) ---
  1. recommend_outfit({"conditions": "humid and partly cloudy", "temperature": 31})   id=fc_2f92bc94-cf92-4ccc-8479-50b096a12524

Check recommend_outfit arguments against the fetched weather:
  temperature: model=31       fetched=31     -> REUSED
  conditions : model='humid and partly cloudy' fetched='humid and partly cloudy' -> REUSED


In [23]:
if ai_round2 is not None and ai_round2.tool_calls:
    round2_results = execute_tool_calls(ai_round2, tools_4, messages_4)

--- Manually executed tool results ---
  recommend_outfit -> {"temperature": 31.0, "conditions": "humid and partly cloudy", "outfit": "Light cotton or linen t-shirt, breathable shorts or chinos, sandals", "extras": ["sweat-wicking fabric and a water bottle"]}


In [24]:
final_4, _ = get_final_answer(llm_with_tools_4, messages_4, tools_4)

=== Final answer ===
**Today's outfit for Mumbai (31 °C, humid and partly‑cloudy):**

- **Top:** Light cotton or linen t‑shirt (or a breathable short‑sleeve shirt).  
- **Bottom:** Comfortable shorts or lightweight chinos.  
- **Footwear:** Sandals or breathable canvas sneakers.  
- **Extras:** Choose sweat‑wicking fabrics to stay dry, and keep a water bottle handy to stay hydrated.  

Stay cool and enjoy your day!


---
## Exercise 5 — Async Product Stock Checker
**Tool method:** a `BaseTool` subclass with both `_run` (sync) and `_arun` (async).

We first call the tool directly (no LLM) with `.invoke()` and `.ainvoke()`, then run the full `bind_tools()` flow.
Jupyter supports top-level `await`, so `await tool.ainvoke(...)` works directly in a cell.

In [25]:
INVENTORY = {
    "P1023": {"name": "Wireless Mouse",       "stock": 42,  "warehouse": "Pune"},
    "P1024": {"name": "Mechanical Keyboard",  "stock": 0,   "warehouse": "Mumbai"},
    "P1025": {"name": "27-inch Monitor",      "stock": 7,   "warehouse": "Bengaluru"},
    "P1026": {"name": "USB-C Hub",            "stock": 120, "warehouse": "Delhi"},
}

class StockCheckInput(BaseModel):
    product_id: str = Field(description="Product ID, e.g. P1023")

class ProductStockTool(BaseTool):
    name: str = "check_product_stock"
    description: str = "Check whether a product is in stock (and how many units) given its product ID."
    args_schema: Type[BaseModel] = StockCheckInput

    def _lookup(self, product_id: str) -> dict:
        pid = product_id.strip().upper()
        item = INVENTORY.get(pid)
        if item is None:
            return {"product_id": pid, "found": False}
        return {"product_id": pid, "found": True, "name": item["name"], "in_stock": item["stock"] > 0,
                "units_available": item["stock"], "warehouse": item["warehouse"]}

    def _run(self, product_id: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> dict:
        time.sleep(0.5)                     # simulates a blocking database / API call
        return {**self._lookup(product_id), "mode": "sync"}

    async def _arun(self, product_id: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None) -> dict:
        await asyncio.sleep(0.5)            # simulates a non-blocking (async) database / API call
        return {**self._lookup(product_id), "mode": "async"}

stock_tool = ProductStockTool()
print("Tool:", stock_tool.name, "| args:", stock_tool.args)

Tool: check_product_stock | args: {'product_id': {'description': 'Product ID, e.g. P1023', 'title': 'Product Id', 'type': 'string'}}


In [26]:
# Manual SYNC call  ->  .invoke()  runs _run
product_id = "P1023"
t0 = time.perf_counter()
sync_result = stock_tool.invoke({"product_id": product_id})
print(f"Sync result  ({time.perf_counter() - t0:.2f}s):", sync_result)

Sync result  (0.50s): {'product_id': 'P1023', 'found': True, 'name': 'Wireless Mouse', 'in_stock': True, 'units_available': 42, 'warehouse': 'Pune', 'mode': 'sync'}


In [27]:
# Manual ASYNC call  ->  .ainvoke()  runs _arun
t0 = time.perf_counter()
async_result = await stock_tool.ainvoke({"product_id": product_id})
print(f"Async result ({time.perf_counter() - t0:.2f}s):", async_result)

Async result (0.51s): {'product_id': 'P1023', 'found': True, 'name': 'Wireless Mouse', 'in_stock': True, 'units_available': 42, 'warehouse': 'Pune', 'mode': 'async'}


In [28]:
# Why async matters: 4 lookups one after another (sync) vs. concurrently (async)
ids = list(INVENTORY)

t0 = time.perf_counter()
seq = [stock_tool.invoke({"product_id": p}) for p in ids]
t_sync = time.perf_counter() - t0

t0 = time.perf_counter()
conc = await asyncio.gather(*[stock_tool.ainvoke({"product_id": p}) for p in ids])
t_async = time.perf_counter() - t0

print(f"4 sync calls   : {t_sync:.2f} s")
print(f"4 async calls  : {t_async:.2f} s  (run concurrently with asyncio.gather)")
display(pd.DataFrame(conc)[["product_id", "name", "in_stock", "units_available", "warehouse"]])

4 sync calls   : 2.00 s
4 async calls  : 0.51 s  (run concurrently with asyncio.gather)


,product_id,name,in_stock,units_available,warehouse
0,P1023,Wireless Mouse,True,42,Pune
1,P1024,Mechanical Keyboard,False,0,Mumbai
2,P1025,27-inch Monitor,True,7,Bengaluru
3,P1026,USB-C Hub,True,120,Delhi


In [29]:
# Full bind_tools() flow for a natural-language question (async end-to-end)
tools_5 = [stock_tool]
llm_with_tools_5 = llm.bind_tools(tools_5)

user_query = "Is product P1023 in stock?"
messages_5 = [HumanMessage(user_query)]

ai_msg_5 = await llm_with_tools_5.ainvoke(messages_5)
messages_5.append(ai_msg_5)
show_tool_calls(ai_msg_5)

print("--- Manually executed tool results (async) ---")
for tc in ai_msg_5.tool_calls:
    result = await stock_tool.ainvoke(tc["args"])
    print(f"  {tc['name']} -> {result}")
    messages_5.append(ToolMessage(content=json.dumps(result), tool_call_id=tc["id"], name=tc["name"]))

final_5 = await llm_with_tools_5.ainvoke(messages_5)
print("=== Final answer ===")
print(text_of(final_5))

--- Model's suggested tool_calls (1) ---
  1. check_product_stock({"product_id": "P1023"})   id=fc_02b93a6a-fcb2-4e15-8545-1a30abb3eac5
--- Manually executed tool results (async) ---


  check_product_stock -> {'product_id': 'P1023', 'found': True, 'name': 'Wireless Mouse', 'in_stock': True, 'units_available': 42, 'warehouse': 'Pune', 'mode': 'async'}


=== Final answer ===
Yes, product **P1023 (Wireless Mouse)** is in stock. There are **42 units** available in the Pune warehouse.


---
## Exercise 6 — Expense Splitter Tool
**Tool method:** `StructuredTool` + Pydantic `args_schema` (`total_amount > 0`, `num_people > 0`).

We call the tool **manually** first, then through the **bound model**, and confirm that both give the same result.

In [30]:
class SplitInput(BaseModel):
    total_amount: float = Field(gt=0, description="Total bill amount in rupees")
    num_people: int = Field(gt=0, description="Number of people sharing the bill")

def split_expense(total_amount: float, num_people: int) -> dict:
    share = round(total_amount / num_people, 2)
    return {"total_amount": total_amount, "num_people": num_people, "amount_per_person": share,
            "rounding_difference": round(total_amount - share * num_people, 2)}

split_tool = StructuredTool.from_function(
    func=split_expense,
    name="split_expense",
    description="Split a total bill equally among a number of people and return how much each person owes.",
    args_schema=SplitInput,
)

# Manual call (no model)
total_amount, num_people = 2400, 4
manual_result = split_tool.invoke({"total_amount": total_amount, "num_people": num_people})
print("Manual call result:", manual_result)

Manual call result: {'total_amount': 2400.0, 'num_people': 4, 'amount_per_person': 600.0, 'rounding_difference': 0.0}


In [31]:
tools_6 = [split_tool]
llm_with_tools_6 = llm.bind_tools(tools_6)

user_query = "Split ₹2400 between 4 friends"
messages_6 = [HumanMessage(user_query)]

ai_msg_6 = llm_with_tools_6.invoke(messages_6)
messages_6.append(ai_msg_6)
show_tool_calls(ai_msg_6)

--- Model's suggested tool_calls (1) ---
  1. split_expense({"num_people": 4, "total_amount": 2400})   id=fc_52d253e1-6625-4395-b8e5-f07b0ec867aa


In [32]:
results_6 = execute_tool_calls(ai_msg_6, tools_6, messages_6)
bound_result = results_6[0]["result"] if results_6 else None

--- Manually executed tool results ---
  split_expense -> {"total_amount": 2400.0, "num_people": 4, "amount_per_person": 600.0, "rounding_difference": 0.0}


In [33]:
final_6, _ = get_final_answer(llm_with_tools_6, messages_6, tools_6)

=== Final answer ===
Each friend should pay **₹ 600**.


In [34]:
# Compare: manual call vs. bound-model flow
comparison = pd.DataFrame({"manual call": manual_result, "bound-model flow": bound_result})
display(comparison)
print("Results identical?", "YES" if manual_result == bound_result else "NO")

,manual call,bound-model flow
total_amount,2400.0,2400.0
num_people,4.0,4.0
amount_per_person,600.0,600.0
rounding_difference,0.0,0.0


Results identical? YES


---
## Exercise 7 — Trip Cost Estimator (Multiple Independent Tools, Single Query)
**Tool method:** `@tool` for all three tools. One query should make the model call **all three** tools. Since the calls don't depend on each other, the model can request them together in one response.

In [35]:
FLIGHT_FARES = {("delhi", "goa"): 6500, ("mumbai", "goa"): 3800, ("pune", "goa"): 4200,
                ("delhi", "mumbai"): 5200, ("bangalore", "goa"): 4500}
HOTEL_RATES = {"goa": 3500, "mumbai": 5000, "delhi": 4000, "jaipur": 3000, "manali": 2800}   # per night
FOOD_PER_DAY = {"goa": 1500, "mumbai": 1800, "delhi": 1400, "jaipur": 1100, "manali": 1000}  # per person

@tool
def get_flight_cost(origin: str, destination: str) -> dict:
    '''Get the approximate round-trip economy flight fare per person between two cities (INR).

    Args:
        origin: Departure city.
        destination: Arrival city.
    '''
    key = (origin.strip().lower(), destination.strip().lower())
    fare_one_way = FLIGHT_FARES.get(key) or FLIGHT_FARES.get(key[::-1])
    if fare_one_way is None:
        return {"origin": origin, "destination": destination, "error": "No fare data"}
    return {"origin": origin.title(), "destination": destination.title(),
            "one_way_inr": fare_one_way, "round_trip_inr": fare_one_way * 2}

@tool
def get_hotel_cost(city: str, nights: int) -> dict:
    '''Get the approximate cost of a mid-range hotel room in a city for a number of nights (INR).

    Args:
        city: City where the hotel is.
        nights: Number of nights to stay.
    '''
    rate = HOTEL_RATES.get(city.strip().lower())
    if rate is None:
        return {"city": city, "error": "No hotel data"}
    return {"city": city.title(), "nights": nights, "rate_per_night_inr": rate, "total_inr": rate * nights}

@tool
def get_food_budget(city: str, days: int) -> dict:
    '''Get an approximate food budget per person for a number of days in a city (INR).

    Args:
        city: City of the trip.
        days: Number of days.
    '''
    per_day = FOOD_PER_DAY.get(city.strip().lower())
    if per_day is None:
        return {"city": city, "error": "No food data"}
    return {"city": city.title(), "days": days, "per_day_inr": per_day, "total_inr": per_day * days}

print(get_flight_cost.invoke({"origin": "Delhi", "destination": "Goa"}))
print(get_hotel_cost.invoke({"city": "Goa", "nights": 3}))
print(get_food_budget.invoke({"city": "Goa", "days": 3}))

{'origin': 'Delhi', 'destination': 'Goa', 'one_way_inr': 6500, 'round_trip_inr': 13000}
{'city': 'Goa', 'nights': 3, 'rate_per_night_inr': 3500, 'total_inr': 10500}
{'city': 'Goa', 'days': 3, 'per_day_inr': 1500, 'total_inr': 4500}


In [36]:
tools_7 = [get_flight_cost, get_hotel_cost, get_food_budget]
llm_with_tools_7 = llm.bind_tools(tools_7)          # all three bound in one bind_tools() call

user_query = "I'm planning a 3-night trip from Delhi to Goa. What will flights, hotel, and food roughly cost?"
messages_7 = [HumanMessage(user_query)]

ai_msg_7 = llm_with_tools_7.invoke(messages_7)
messages_7.append(ai_msg_7)
show_tool_calls(ai_msg_7)
print("\nNumber of tool calls:", len(ai_msg_7.tool_calls), "(expected 3)")

--- Model's suggested tool_calls (1) ---
  1. get_flight_cost({"destination": "Goa", "origin": "Delhi"})   id=fc_60eb5a3e-213b-4b89-8bf3-29d7e6ab4dab

Number of tool calls: 1 (expected 3)


In [37]:
results_7 = execute_tool_calls(ai_msg_7, tools_7, messages_7)

--- Manually executed tool results ---
  get_flight_cost -> {"origin": "Delhi", "destination": "Goa", "one_way_inr": 6500, "round_trip_inr": 13000}


In [38]:
final_7, extra_7 = get_final_answer(llm_with_tools_7, messages_7, tools_7)

--- Model asked for more tools (follow-up round 1) (1) ---
  1. get_hotel_cost({"city": "Goa", "nights": 3})   id=fc_1e5e72dc-2c8e-406d-8ee9-816ae10be022
--- Manually executed tool results ---
  get_hotel_cost -> {"city": "Goa", "nights": 3, "rate_per_night_inr": 3500, "total_inr": 10500}


--- Model asked for more tools (follow-up round 2) (1) ---
  1. get_food_budget({"city": "Goa", "days": 3})   id=fc_2fce311a-04f3-4e56-95ea-7b127db22d9b
--- Manually executed tool results ---
  get_food_budget -> {"city": "Goa", "days": 3, "per_day_inr": 1500, "total_inr": 4500}


=== Final answer ===
Here’s a quick, ball‑park estimate for a **3‑night (4‑day) trip from Delhi to Goa** for one person:

| Item | Details | Approx. Cost (INR) |
|------|---------|--------------------|
| **Flights** | Round‑trip (Delhi ↔ Goa) – economy class, average fare | **13,000** |
| **Hotel** | Mid‑range hotel, 3 nights (≈ ₹3,500 per night) | **10,500** |
| **Food** | Meals & snacks, 3 days (≈ ₹1,500 per day) | **4,500** |
| **Total (per person)** |  | **≈ ₹28,000** |

### How the numbers were derived
- **Flights:** Current average round‑trip economy fare between Delhi and Goa is about **₹13,000** per person.
- **Hotel:** A typical mid‑range property in popular areas (e.g., Calangute, Panaji) costs around **₹3,500 per night**. For 3 nights that’s **₹10,500**.
- **Food:** Assuming a mix of breakfast (often included in the hotel), lunch, dinner and occasional snacks, a comfortable daily food budget is roughly **₹1,500**. Over 3 days that totals **₹4,500**.

### Quick tips to keep c

---
## Exercise 8 — Health Metrics Toolkit (BMI + Calorie Needs)
**Tool method:** two `StructuredTool`s grouped into a `HealthToolkit` class with a `get_tools()` method (it subclasses LangChain's `BaseToolkit`).

* **BMI** = weight (kg) / height (m)².
* **Daily calories**: Mifflin-St Jeor BMR × activity factor. The required inputs don't include sex, so the tool uses the average of the male (+5) and female (−161) constants (−78). This gives a sex-neutral estimate.

In [39]:
class BMIInput(BaseModel):
    weight_kg: float = Field(gt=0, le=400, description="Body weight in kilograms")
    height_cm: float = Field(gt=50, le=260, description="Height in centimetres")

class CalorieInput(BaseModel):
    weight_kg: float = Field(gt=0, le=400, description="Body weight in kilograms")
    height_cm: float = Field(gt=50, le=260, description="Height in centimetres")
    age: int = Field(gt=0, lt=120, description="Age in years")
    activity_level: Literal["sedentary", "light", "moderate", "active", "very_active"] = Field(
        description="Activity level: sedentary, light, moderate, active or very_active")

def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    bmi = weight_kg / (height_cm / 100) ** 2
    category = ("Underweight" if bmi < 18.5 else "Normal weight" if bmi < 25
                else "Overweight" if bmi < 30 else "Obese")
    return {"bmi": round(bmi, 1), "category": category}

ACTIVITY_FACTORS = {"sedentary": 1.2, "light": 1.375, "moderate": 1.55, "active": 1.725, "very_active": 1.9}

def calculate_daily_calories(weight_kg: float, height_cm: float, age: int, activity_level: str) -> dict:
    bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age - 78          # Mifflin-St Jeor, sex-neutral constant
    tdee = bmr * ACTIVITY_FACTORS[activity_level]
    return {"bmr_kcal": round(bmr), "activity_level": activity_level,
            "activity_factor": ACTIVITY_FACTORS[activity_level], "daily_calories_kcal": round(tdee)}

class HealthToolkit(BaseToolkit):
    # Groups the health tools so they can be bound together
    def get_tools(self):
        return [
            StructuredTool.from_function(func=calculate_bmi, name="calculate_bmi",
                description="Calculate Body Mass Index (BMI) and its category from weight and height.",
                args_schema=BMIInput),
            StructuredTool.from_function(func=calculate_daily_calories, name="calculate_daily_calories",
                description="Estimate daily calorie needs (maintenance kcal) from weight, height, age and activity level.",
                args_schema=CalorieInput),
        ]

toolkit = HealthToolkit()
tools_8 = toolkit.get_tools()
print("Toolkit tools:")
for t in tools_8:
    print(f"  - {t.name}: {t.description}\n      args: {list(t.args)}")

Toolkit tools:
  - calculate_bmi: Calculate Body Mass Index (BMI) and its category from weight and height.
      args: ['weight_kg', 'height_cm']
  - calculate_daily_calories: Estimate daily calorie needs (maintenance kcal) from weight, height, age and activity level.
      args: ['weight_kg', 'height_cm', 'age', 'activity_level']


In [40]:
llm_with_tools_8 = llm.bind_tools(tools_8)

user_query = "I'm 70kg, 175cm tall, 25 years old, moderately active. What's my BMI and daily calorie need?"
messages_8 = [HumanMessage(user_query)]

ai_msg_8 = llm_with_tools_8.invoke(messages_8)
messages_8.append(ai_msg_8)
show_tool_calls(ai_msg_8)

--- Model's suggested tool_calls (1) ---
  1. calculate_bmi({"height_cm": 175, "weight_kg": 70})   id=fc_f6a16c14-8bf5-4b23-a116-e229665e54d8


In [41]:
results_8 = execute_tool_calls(ai_msg_8, tools_8, messages_8)

--- Manually executed tool results ---
  calculate_bmi -> {"bmi": 22.9, "category": "Normal weight"}


In [42]:
final_8, _ = get_final_answer(llm_with_tools_8, messages_8, tools_8)

--- Model asked for more tools (follow-up round 1) (1) ---
  1. calculate_daily_calories({"activity_level": "moderate", "age": 25, "height_cm": 175, "weight_kg": 70})   id=fc_da704011-55d7-4684-8497-cbf7a9b58b2a
--- Manually executed tool results ---
  calculate_daily_calories -> {"bmr_kcal": 1591, "activity_level": "moderate", "activity_factor": 1.55, "daily_calories_kcal": 2466}


=== Final answer ===
**Your Body Mass Index (BMI)**  
- **BMI:** **22.9**  
- **Category:** Normal weight  

**Your Estimated Daily Calorie Needs (maintenance)**  
- **Basal Metabolic Rate (BMR):** 1,591 kcal/day  
- **Activity factor (moderately active):** 1.55  
- **Total daily calories to maintain weight:** **≈ 2,466 kcal/day**  

---

### How the numbers are calculated  

1. **BMI** = weight (kg) ÷ (height (m))²  
   \[
   \text{BMI} = \frac{70}{(1.75)^2} \approx 22.9
   \]

2. **BMR** (Mifflin‑St Jeor equation for men, assuming male; if female, the BMR would be slightly lower):
   \[
   \text{BMR} = 10 \times \text{weight (kg)} + 6.25 \times \text{height (cm)} - 5 \times \text{age} + 5
   \]
   \[
   = 10 \times 70 + 6.25 \times 175 - 5 \times 25 + 5 = 1,591 \text{ kcal/day}
   \]

3. **Daily maintenance calories** = BMR × activity factor  
   \[
   1,591 \times 1.55 \approx 2,466 \text{ kcal/day}
   \]

---

### What this means for you  

- **BMI of 22.9** places you comfortably 

---
## Exercise 9 — Smart Meeting Scheduler (Multiple Tools, Partial Use)
**Tool method:** `@tool` for all three tools. All three are bound, and we ask two different queries:
* **Query A (availability only):** the model should call only `check_calendar_availability`.
* **Query B (full booking):** the model should check availability, get the time-zone offset and then book.
  Booking depends on the earlier results, so this can take more than one round.

Finally, the tool calls for both queries are shown side by side.

In [43]:
CALENDAR = {   # free 1-hour slots (local time of the person) per date
    "priya": {"2026-10-28": ["10:00", "14:00", "16:00"], "2026-10-29": ["11:00"]},
    "rahul": {"2026-10-28": ["09:00"], "2026-10-29": ["09:00", "11:00", "15:00"]},
    "anita": {"2026-10-28": [], "2026-10-29": ["13:00", "17:00"]},
}
TIMEZONES = {"pune": "+05:30", "mumbai": "+05:30", "london": "+00:00", "new york": "-04:00",
             "dubai": "+04:00", "singapore": "+08:00", "san francisco": "-07:00"}
BOOKINGS = []

@tool
def check_calendar_availability(person: str, date: str) -> dict:
    '''Check which 1-hour slots a person has free on a date.

    Args:
        person: First name of the person, e.g. "Priya".
        date: Date in YYYY-MM-DD format.
    '''
    slots = CALENDAR.get(person.strip().lower(), {}).get(date)
    if slots is None:
        return {"person": person, "date": date, "error": "No calendar data"}
    return {"person": person.title(), "date": date, "free_slots": slots, "is_free": bool(slots)}

@tool
def get_timezone_offset(city: str) -> dict:
    '''Get the UTC offset of a city's time zone.

    Args:
        city: City name, e.g. "London".
    '''
    off = TIMEZONES.get(city.strip().lower())
    return {"city": city.title(), "utc_offset": off} if off else {"city": city, "error": "Unknown city"}

@tool
def book_meeting(person: str, date: str, time: str) -> dict:
    '''Book a 1-hour meeting with a person. Only works for a slot that is free in their calendar.

    Args:
        person: First name of the person.
        date: Date in YYYY-MM-DD format.
        time: Start time in HH:MM (24-hour, the person's local time).
    '''
    slots = CALENDAR.get(person.strip().lower(), {}).get(date, [])
    if time not in slots:
        return {"booked": False, "reason": f"{time} is not a free slot", "free_slots": slots}
    slots.remove(time)
    booking = {"booked": True, "meeting_id": f"MTG-{len(BOOKINGS) + 1001}", "person": person.title(),
               "date": date, "time": time}
    BOOKINGS.append(booking)
    return booking

tools_9 = [check_calendar_availability, get_timezone_offset, book_meeting]
llm_with_tools_9 = llm.bind_tools(tools_9)
print("Bound tools:", [t.name for t in tools_9])

Bound tools: ['check_calendar_availability', 'get_timezone_offset', 'book_meeting']


In [44]:
# ---------------- Query A: availability only ----------------
query_a = "Is Priya free on 2026-10-28?"
messages_9a = [HumanMessage(query_a)]

ai_9a = llm_with_tools_9.invoke(messages_9a)
messages_9a.append(ai_9a)
show_tool_calls(ai_9a, "Query A - suggested tool_calls")
results_9a = execute_tool_calls(ai_9a, tools_9, messages_9a)
final_9a, extra_9a = get_final_answer(llm_with_tools_9, messages_9a, tools_9)
calls_a = results_9a + extra_9a

--- Query A - suggested tool_calls (1) ---
  1. check_calendar_availability({"date": "2026-10-28", "person": "Priya"})   id=fc_3f06274f-419b-451d-8fce-77a6dddde6dc
--- Manually executed tool results ---
  check_calendar_availability -> {"person": "Priya", "date": "2026-10-28", "free_slots": ["10:00", "14:00", "16:00"], "is_free": true}


=== Final answer ===
Yes—Priya is free on 2026‑10‑28. She has three one‑hour slots available that day:

- **10:00 – 11:00**
- **14:00 – 15:00**
- **16:00 – 17:00**

(These times are in Priya’s local time zone.) Let me know if you’d like to book any of those slots or need anything else.


In [45]:
# ---------------- Query B: full booking (needs all three tools) ----------------
query_b = ("I'm in Pune and need a 1-hour meeting with Rahul, who works from London, on 2026-10-29. "
           "Check when he is free, find the time difference between our cities, then book the earliest free slot.")
messages_9b = [HumanMessage(query_b)]

ai_9b = llm_with_tools_9.invoke(messages_9b)
messages_9b.append(ai_9b)
show_tool_calls(ai_9b, "Query B - round 1 suggested tool_calls")
results_9b = execute_tool_calls(ai_9b, tools_9, messages_9b)
final_9b, extra_9b = get_final_answer(llm_with_tools_9, messages_9b, tools_9)   # runs later rounds (e.g. book_meeting)
calls_b = results_9b + extra_9b

--- Query B - round 1 suggested tool_calls (1) ---
  1. check_calendar_availability({"date": "2026-10-29", "person": "Rahul"})   id=fc_f06c5896-4ef4-4d0b-9c78-f80aa72ef77f
--- Manually executed tool results ---
  check_calendar_availability -> {"person": "Rahul", "date": "2026-10-29", "free_slots": ["09:00", "11:00", "15:00"], "is_free": true}


--- Model asked for more tools (follow-up round 1) (1) ---
  1. get_timezone_offset({"city": "Pune"})   id=fc_b5501d14-f360-47a0-80da-f54a790b505a
--- Manually executed tool results ---
  get_timezone_offset -> {"city": "Pune", "utc_offset": "+05:30"}


--- Model asked for more tools (follow-up round 2) (1) ---
  1. get_timezone_offset({"city": "London"})   id=fc_f24ec545-f476-4206-a60f-e6e59222de43
--- Manually executed tool results ---
  get_timezone_offset -> {"city": "London", "utc_offset": "+00:00"}


--- Model asked for more tools (follow-up round 3) (1) ---
  1. book_meeting({"date": "2026-10-29", "person": "Rahul", "time": "09:00"})   id=fc_b6af1fcb-2809-4e6b-82f4-cbc37a105705
--- Manually executed tool results ---
  book_meeting -> {"booked": true, "meeting_id": "MTG-1001", "person": "Rahul", "date": "2026-10-29", "time": "09:00"}


=== Final answer ===
Rahul’s calendar on 2026‑10‑29 shows three free 1‑hour slots: **09:00, 11:00, 15:00** (London local time).  

- Pune is UTC +5:30, London is UTC +0:00, so Pune is **5 hours 30 minutes ahead** of London.  
- Converting the earliest free slot (09:00 London) to Pune time gives **14:30 – 15:30** Pune time.

I’ve booked the earliest available slot:

**Meeting ID:** MTG‑1001  
**Date:** 2026‑10‑29  
**Time:** 09:00 – 10:00 (London time) / 14:30 – 15:30 (Pune time)

Let me know if you need any adjustments!


In [46]:
# Side-by-side comparison of the tool calls made for each query
fmt = lambda c: f"{c['tool']}({', '.join(f'{k}={v!r}' for k, v in c['args'].items())})"
rows = max(len(calls_a), len(calls_b))
side_by_side = pd.DataFrame({
    "Query A: availability only": [fmt(c) for c in calls_a] + [""] * (rows - len(calls_a)),
    "Query B: full booking":      [fmt(c) for c in calls_b] + [""] * (rows - len(calls_b)),
}, index=range(1, rows + 1))
display(side_by_side)

summary = pd.DataFrame({
    "tools bound": [len(tools_9), len(tools_9)],
    "tool calls made": [len(calls_a), len(calls_b)],
    "distinct tools used": [sorted({c["tool"] for c in calls_a}), sorted({c["tool"] for c in calls_b})],
}, index=["Query A", "Query B"])
display(summary)
print("Bookings in calendar:", BOOKINGS)

,Query A: availability only,Query B: full booking
1,"check_calendar_availability(date='2026-10-28', person='Priya')","check_calendar_availability(date='2026-10-29', person='Rahul')"
2,,get_timezone_offset(city='Pune')
3,,get_timezone_offset(city='London')
4,,"book_meeting(date='2026-10-29', person='Rahul', time='09:00')"


,tools bound,tool calls made,distinct tools used
Query A,3,1,[check_calendar_availability]
Query B,3,4,"[book_meeting, check_calendar_availability, get_timezone_offset]"


Bookings in calendar: [{'booked': True, 'meeting_id': 'MTG-1001', 'person': 'Rahul', 'date': '2026-10-29', 'time': '09:00'}]


---
## Exercise 10 — Restaurant Recommendation Assistant (4+ Tools Bound Together)
**Tool method:** `@tool` for all four tools. They are all bound in one `bind_tools()` call. The query needs several of them: first find restaurants, then check rating, opening day and cost for each one.

A short system message gives the model today's day of the week, so it can check "open today".

In [47]:
RESTAURANTS = {
    "Italian Kitchen Koregaon Park": {"city": "pune", "cuisine": "italian", "rating": 4.5,
                                      "closed_on": ["Monday"], "cost_for_two": 2200},
    "La Pizzeria Baner":             {"city": "pune", "cuisine": "italian", "rating": 4.1,
                                      "closed_on": ["Tuesday"], "cost_for_two": 1400},
    "Little Italy FC Road":          {"city": "pune", "cuisine": "italian", "rating": 4.3,
                                      "closed_on": ["Wednesday"], "cost_for_two": 1600},
    "Malaka Spice":                  {"city": "pune", "cuisine": "asian", "rating": 4.4,
                                      "closed_on": [], "cost_for_two": 1800},
    "Pizza Express Bandra":          {"city": "mumbai", "cuisine": "italian", "rating": 4.2,
                                      "closed_on": [], "cost_for_two": 2000},
}

def _find(name):
    for n, d in RESTAURANTS.items():
        if n.lower() == name.strip().lower():
            return n, d
    return name, None

@tool
def find_restaurants(cuisine: str, city: str) -> dict:
    '''Find restaurants serving a cuisine in a city. Returns only the restaurant names.

    Args:
        cuisine: Cuisine type, e.g. "Italian".
        city: City name, e.g. "Pune".
    '''
    names = [n for n, d in RESTAURANTS.items()
             if d["cuisine"] == cuisine.strip().lower() and d["city"] == city.strip().lower()]
    return {"cuisine": cuisine, "city": city, "restaurants": names}

@tool
def get_restaurant_rating(restaurant_name: str) -> dict:
    '''Get the customer rating of a restaurant (out of 5).

    Args:
        restaurant_name: Exact restaurant name as returned by find_restaurants.
    '''
    n, d = _find(restaurant_name)
    return {"restaurant": n, "rating_out_of_5": d["rating"]} if d else {"restaurant": n, "error": "Not found"}

@tool
def check_restaurant_open(restaurant_name: str, day: str) -> dict:
    '''Check whether a restaurant is open on a given day of the week.

    Args:
        restaurant_name: Exact restaurant name as returned by find_restaurants.
        day: Day of the week, e.g. "Wednesday".
    '''
    n, d = _find(restaurant_name)
    if not d:
        return {"restaurant": n, "error": "Not found"}
    return {"restaurant": n, "day": day.title(), "is_open": day.title() not in d["closed_on"]}

@tool
def estimate_cost_for_two(restaurant_name: str) -> dict:
    '''Get the approximate cost for two people at a restaurant (INR).

    Args:
        restaurant_name: Exact restaurant name as returned by find_restaurants.
    '''
    n, d = _find(restaurant_name)
    return {"restaurant": n, "approx_cost_for_two_inr": d["cost_for_two"]} if d else {"restaurant": n, "error": "Not found"}

tools_10 = [find_restaurants, get_restaurant_rating, check_restaurant_open, estimate_cost_for_two]
llm_with_tools_10 = llm.bind_tools(tools_10)         # four tools bound in a single call
print("Bound tools:", [t.name for t in tools_10])

Bound tools: ['find_restaurants', 'get_restaurant_rating', 'check_restaurant_open', 'estimate_cost_for_two']


In [48]:
today = datetime.now().strftime("%A")
user_query = "Suggest a good Italian restaurant in Pune that's open today, with its rating and rough cost for two"
messages_10 = [
    SystemMessage(f"You are a restaurant assistant. Today is {today}. Use the tools to get real data; never guess."),
    HumanMessage(user_query),
]

ai_msg_10 = llm_with_tools_10.invoke(messages_10)
messages_10.append(ai_msg_10)
show_tool_calls(ai_msg_10, "Round 1 - suggested tool_calls")

--- Round 1 - suggested tool_calls (1) ---
  1. find_restaurants({"city": "Pune", "cuisine": "Italian"})   id=fc_a75ad7bd-d9c6-45fc-9aa7-17821edea160


In [49]:
results_10 = execute_tool_calls(ai_msg_10, tools_10, messages_10)

--- Manually executed tool results ---
  find_restaurants -> {"cuisine": "Italian", "city": "Pune", "restaurants": ["Italian Kitchen Koregaon Park", "La Pizzeria Baner", "Little Italy FC Road"]}


In [50]:
# Feeds results back; the model will usually now ask for rating / open-check / cost for the restaurants found
final_10, extra_10 = get_final_answer(llm_with_tools_10, messages_10, tools_10, max_rounds=10)

--- Model asked for more tools (follow-up round 1) (1) ---
  1. check_restaurant_open({"day": "Wednesday", "restaurant_name": "Italian Kitchen Koregaon Park"})   id=fc_1773bb02-80b7-4297-b94a-64fc4c8544c9
--- Manually executed tool results ---
  check_restaurant_open -> {"restaurant": "Italian Kitchen Koregaon Park", "day": "Wednesday", "is_open": true}


--- Model asked for more tools (follow-up round 2) (1) ---
  1. get_restaurant_rating({"restaurant_name": "Italian Kitchen Koregaon Park"})   id=fc_1506b354-2d4d-4723-8fe8-07e50623fd5e
--- Manually executed tool results ---
  get_restaurant_rating -> {"restaurant": "Italian Kitchen Koregaon Park", "rating_out_of_5": 4.5}


--- Model asked for more tools (follow-up round 3) (1) ---
  1. estimate_cost_for_two({"restaurant_name": "Italian Kitchen Koregaon Park"})   id=fc_a515b9da-4b61-4fb3-8ba6-8f780323b308
--- Manually executed tool results ---
  estimate_cost_for_two -> {"restaurant": "Italian Kitchen Koregaon Park", "approx_cost_for_two_inr": 2200}


=== Final answer ===
**Italian Kitchen – Koregaon Park, Pune**

- **Open today (Wednesday):** Yes  
- **Customer rating:** 4.5 / 5  
- **Approximate cost for two:** ₹2,200  

Italian Kitchen offers a cozy ambience with a solid menu of classic Italian dishes—think wood‑fired pizzas, creamy pastas, and tasty antipasti. The high rating reflects consistent food quality and good service, and the price point is moderate for a nice dinner for two. Enjoy your meal!


In [51]:
# Summary of every tool call the model made in Exercise 10
all_10 = results_10 + extra_10
display(pd.DataFrame([{"tool": c["tool"], "args": c["args"], "result": c["result"]} for c in all_10]))
print("Total tool calls:", len(all_10), "| distinct tools used:", sorted({c["tool"] for c in all_10}))

,tool,args,result
0,find_restaurants,"{'city': 'Pune', 'cuisine': 'Italian'}","{'cuisine': 'Italian', 'city': 'Pune', 'restaurants': ['Italian Kitchen Koregaon Park', 'La Pizzeria Baner', 'Little Italy FC Road']}"
1,check_restaurant_open,"{'day': 'Wednesday', 'restaurant_name': 'Italian Kitchen Koregaon Park'}","{'restaurant': 'Italian Kitchen Koregaon Park', 'day': 'Wednesday', 'is_open': True}"
2,get_restaurant_rating,{'restaurant_name': 'Italian Kitchen Koregaon Park'},"{'restaurant': 'Italian Kitchen Koregaon Park', 'rating_out_of_5': 4.5}"
3,estimate_cost_for_two,{'restaurant_name': 'Italian Kitchen Koregaon Park'},"{'restaurant': 'Italian Kitchen Koregaon Park', 'approx_cost_for_two_inr': 2200}"


Total tool calls: 4 | distinct tools used: ['check_restaurant_open', 'estimate_cost_for_two', 'find_restaurants', 'get_restaurant_rating']


---
## Summary
| Ex. | Tool(s) | Method | What it shows |
|---|---|---|---|
| 1 | `get_order_status` | `@tool` | Single tool: suggested call → manual execution → final answer |
| 2 | `calculate_emi` | `StructuredTool` + Pydantic | Strict input validation (invalid inputs rejected) |
| 3 | `convert_temperature`, `convert_distance` | `@tool` | Model picks the correct tool out of several |
| 4 | `get_weather` → `recommend_outfit` | `@tool` | Chained, multi-round calls; checks reuse of fetched data |
| 5 | `check_product_stock` | `BaseTool` (`_run` + `_arun`) | `.invoke()` vs `.ainvoke()`, concurrency with `asyncio.gather` |
| 6 | `split_expense` | `StructuredTool` + Pydantic | Manual call == bound-model result |
| 7 | flight / hotel / food | `@tool` | Three independent calls from one query |
| 8 | `HealthToolkit` (BMI + calories) | `StructuredTool` in a `BaseToolkit` | Grouping tools into a toolkit |
| 9 | calendar / timezone / booking | `@tool` | Model uses only the tools relevant to each query |
| 10 | 4 restaurant tools | `@tool` | Many tools bound; multi-step reasoning to one recommendation |